# Final Data Load & KPI Preparation

**Objective:** Prepare dashboard-ready dataset and KPI summary for Tableau visualization.

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Load Data
df = pd.read_csv("../data/processed/cleaned_banking_churn.csv")

In [ ]:
# Step 2: KPI Computation
total_customers = len(df)
churn_rate = round(df["churn"].mean() * 100, 2)
active_customer_rate = round((df["is_active"].sum() / total_customers) * 100, 2)
avg_balance = round(df["current_balance"].mean(), 2)
avg_balance_diff = round(df["balance_diff"].mean(), 2)
avg_tenure = round(df["customer_tenure_years"].mean(), 2)

# Create KPI summary
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Churn Rate %",
        "Active Customer Rate %",
        "Average Balance",
        "Average Balance Diff",
        "Average Tenure (Years)"
    ],
    "Value": [
        total_customers,
        churn_rate,
        active_customer_rate,
        avg_balance,
        avg_balance_diff,
        avg_tenure
    ]
})

In [ ]:
# Step 3: Business Columns
df["churn_label"] = df["churn"].map({0: "Retained", 1: "Churned"})
df["activity_status"] = df["is_active"].map({0: "Inactive", 1: "Active"})
df["age_group"] = pd.cut(df["age"], bins=[0,25,35,45,55,65,100], labels=["<=25","26-35","36-45","46-55","56-65","65+"])
df["balance_segment"] = pd.qcut(df["current_balance"], q=4, labels=["Low","Medium","High","Premium"])

In [ ]:
# Step 4: Risk Segmentation
def assign_risk(row):
    if row["is_active"] == 0 and row["balance_diff"] < 0:
        return "High Risk"
    elif row["is_active"] == 0 or row["balance_diff"] < 0:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_segment"] = df.apply(assign_risk, axis=1)

# Step 5: Final Dataset
final_columns = [
    "age",
    "age_group",
    "gender",
    "occupation",
    "current_balance",
    "balance_diff",
    "customer_tenure_years",
    "is_active",
    "churn",
    "churn_label",
    "activity_status",
    "risk_segment"
]
df = df[final_columns]

In [ ]:
# Step 6: Export Files
df.to_csv("../data/processed/final_dashboard_dataset.csv", index=False)
kpi_summary.to_csv("../reports/kpi_summary.csv", index=False)

# Step 7: Validation
print("Shape:", df.shape)
print("Null values:", df.isnull().sum().sum())

In [ ]:
# Step 8: Final Summary
print(f"Total Customers: {total_customers:,}")
print(f"Churn Rate: {churn_rate}%")
print(f"Active Customer Rate: {active_customer_rate}%")